In [1]:
# Install vLLM. Only torchaudio is removed (CUDA mismatch); torchvision must stay.
!pip install vllm httpx
!pip uninstall -y torchaudio

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.9/87.9 kB 5.6 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of cuda-python to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 316.0/316.0 MB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.7/2.7 MB 92.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.7/211.7 kB 23.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.3/18.3 MB 70.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 322.7/322.7 kB 33.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 111.0/111.0 kB 14.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.4/45.4 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.8/3.8 MB 117.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 782.6/782.6 kB 51.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 1

Found existing installation: torchaudio 2.11.0+cu128
Uninstalling torchaudio-2.11.0+cu128:
  Successfully uninstalled torchaudio-2.11.0+cu128


In [1]:
!nvidia-smi --query-gpu=name,memory.used,memory.total --format=csv
!python3 -c "import torch, torchvision, vllm; print('OK', vllm.__version__, torch.__version__, torchvision.__version__)"

name, memory.used [MiB], memory.total [MiB]
Tesla T4, 0 MiB, 15360 MiB
OK 0.29.0 2.13.0+cu130 0.28.0+cu130


In [3]:
!ls -la *.json *.py

-rw-r--r-- 1 root root 198485 Sep 18 22:51 benchmark_payloads.json
-rw-r--r-- 1 root root   9874 Sep 18 22:51 concurrency_benchmark.py
-rw-r--r-- 1 root root   9319 Sep 18 22:51 smoke_test_request.json


In [4]:
from pathlib import Path

p = Path("concurrency_benchmark.py")
src = p.read_text(encoding="utf-8")

if "--files-per-course" in src:
    print("Already patched, nothing to do.")
else:
    # 1) New CLI arguments
    old = "    args = ap.parse_args()"
    new = '''    ap.add_argument("--files-per-course", type=int, default=1,
                    help="Average number of content files (requests) per course")
    ap.add_argument("--gpu-hours-per-day", type=float, default=24,
                    help="Hours per day the GPU is billed (24 = always-on instance)")
    args = ap.parse_args()'''
    assert old in src, "parse_args line not found"
    src = src.replace(old, new)

    # 2) Warm-up with a short prompt (does not pre-fill the prefix cache with real payloads)
    old = "    table_rows = []\n"
    new = '''    async with httpx.AsyncClient() as client:
        for _ in range(3):
            await send_one(client, args.base_url, args.model, "Say hi")
    table_rows = []
'''
    assert old in src, "table_rows line not found"
    src = src.replace(old, new, 1)

    # 3) Replace the cost block (Part 22)
    start = src.index('    print("PART 22')
    end = src.index("    # ---- write CSV ----")
    cost_block = '''    print("PART 22 -- Cost per course (self-hosted GPU)")
    valid = [r for r in table_rows if r["throughput_req_s"]]
    if valid:
        best = max(valid, key=lambda r: r["throughput_req_s"])
        req_per_hour = best["throughput_req_s"] * 3600
        cost_per_request = args.gpu_hourly_cost / req_per_hour
        cost_per_course = cost_per_request * args.files_per_course
        usage_monthly = cost_per_course * args.courses_per_month
        always_on_monthly = args.gpu_hourly_cost * args.gpu_hours_per_day * 30

        print(f"Best throughput at concurrency {best['concurrency']}: "
              f"{best['throughput_req_s']} req/s (~{req_per_hour:.0f} req/hr)")
        print(f"GPU hourly cost:                ${args.gpu_hourly_cost:.2f}/hr")
        print(f"Cost per request:               ${cost_per_request:.5f}")
        print(f"Files per course (assumed):     {args.files_per_course}")
        print(f"Cost per course:                ${cost_per_course:.5f}")
        print(f"Monthly, pay-per-use ({args.courses_per_month} courses): ${usage_monthly:.2f}")
        print(f"Monthly, GPU billed {args.gpu_hours_per_day:g}h/day:     ${always_on_monthly:.2f}")
        print("NOTE: latency at the best-throughput level is "
              f"{best['avg_latency_s']}s avg / {best['p95_latency_s']}s p95.")

'''
    src = src[:start] + cost_block + src[end:]

    p.write_text(src, encoding="utf-8")
    print("Patched OK.")

!python3 -m py_compile concurrency_benchmark.py && echo "Syntax OK"

Patched OK.
Syntax OK


In [5]:
import subprocess, time, httpx, os

MODEL = "Qwen/Qwen2.5-7B-Instruct-AWQ"
EAGER = False            # set True only if startup fails
PREFIX_CACHING = True    # set False to measure without the prefix cache

def ready():
    try:
        return httpx.get("http://localhost:8000/v1/models", timeout=2).status_code == 200
    except Exception:
        return False

if ready():
    print("Server is already running. Skip to the next cell.")
else:
    cmd = ["python3", "-m", "vllm.entrypoints.openai.api_server",
           "--model", MODEL,
           "--quantization", "awq",
           "--dtype", "half",
           "--max-model-len", "16384",
           "--gpu-memory-utilization", "0.85",
           "--port", "8000"]
    if EAGER:
        cmd.append("--enforce-eager")
    if not PREFIX_CACHING:
        cmd.append("--no-enable-prefix-caching")

    env = dict(os.environ, VLLM_USE_FLASHINFER_SAMPLER="0")
    p = subprocess.Popen(cmd, stdout=open("vllm.log", "w"),
                         stderr=subprocess.STDOUT, env=env, start_new_session=True)

    for i in range(60):
        if p.poll() is not None:
            print("Server exited early, exit code:", p.returncode)
            print(subprocess.run("grep 'core.py:1374' vllm.log | tail -6 | cut -c60-400",
                                 shell=True, capture_output=True, text=True).stdout)
            break
        if ready():
            print("Server ready after", i * 10, "seconds")
            break
        time.sleep(10)
    else:
        print("Timed out waiting for the server")

Server ready after 270 seconds


In [6]:
!sed -i 's#Qwen/Qwen2.5-7B-Instruct"#Qwen/Qwen2.5-7B-Instruct-AWQ"#' smoke_test_request.json
!curl -s -X POST http://localhost:8000/v1/chat/completions \
  -H "Content-Type: application/json" \
  -d @smoke_test_request.json | python3 -m json.tool

{
    "id": "chatcmpl-956a8479e0bfba87",
    "object": "chat.completion",
    "created": 1789772209,
    "model": "Qwen/Qwen2.5-7B-Instruct-AWQ",
    "choices": [
        {
            "index": 0,
            "message": {
                "role": "assistant",
                "content": "```json\n{\n\"predicted_tags\": [\"Decision Trees\", \"Gini Impurity\", \"Pruning\"],\n\"difficulty_level\": \"Intermediate\",\n\"confidence\": 0.85,\n\"notes\": \"The content covers the basics of decision trees, including practical implementation, Gini impurity, and pruning techniques. It provides a clear and detailed explanation suitable for intermediate learners.\"\n}\n```",
                "refusal": null,
                "annotations": null,
                "audio": null,
                "function_call": null,
                "reasoning": null
            },
            "logprobs": null,
            "finish_reason": "stop",
            "stop_reason": null,
            "token_ids": null,
            

In [8]:
from pathlib import Path

p = Path("concurrency_benchmark.py")
src = p.read_text(encoding="utf-8")

old = '''        for _ in range(3):
            await send_one(client, args.base_url, args.model, "Say hi")'''

new = '''        # Warm up short AND long prompts (>2048 tokens hits a different compile range)
        warm_prompts = ["Say hi"] + [
            "Summarize the following text in one sentence.\\n\\n"
            + " ".join(f"Sentence {i} discusses topic {i % 7} in a generic way." for i in range(n))
            for n in (300, 600, 1000)   # roughly 3k, 6k, 11k tokens
        ]
        for wp in warm_prompts * 2:
            await send_one(client, args.base_url, args.model, wp)'''

if "warm_prompts" in src:
    print("Already patched.")
else:
    assert old in src, "Old warm-up block not found"
    p.write_text(src.replace(old, new), encoding="utf-8")
    print("Patched OK.")

!python3 -m py_compile concurrency_benchmark.py && echo "Syntax OK"

Patched OK.
Syntax OK


In [9]:
!python3 concurrency_benchmark.py \
  --base-url http://localhost:8000/v1 \
  --model Qwen/Qwen2.5-7B-Instruct-AWQ \
  --payloads benchmark_payloads.json \
  --gpu-hourly-cost 1.80 \
  --courses-per-month 200 \
  --files-per-course 12

Loaded 12 real content payloads (avg ~3914 input tokens each)

GPU before run: 0.0% util, 12123/15360 MB VRAM

--- Concurrency = 1 ---
{
  "n": 20,
  "errors": 0,
  "error_rate_pct": 0.0,
  "avg_latency_s": 3.47,
  "p95_latency_s": 4.516,
  "avg_ttft_s": 0.109,
  "throughput_req_s": 0.288,
  "tokens_per_sec": 30.6,
  "wall_time_s": 69.41,
  "concurrency": 1,
  "gpu_util_pct": 100.0,
  "vram_used_mb": 12123.0
}

--- Concurrency = 2 ---
{
  "n": 20,
  "errors": 0,
  "error_rate_pct": 0.0,
  "avg_latency_s": 4.123,
  "p95_latency_s": 5.685,
  "avg_ttft_s": 0.152,
  "throughput_req_s": 0.468,
  "tokens_per_sec": 49.4,
  "wall_time_s": 42.78,
  "concurrency": 2,
  "gpu_util_pct": 100.0,
  "vram_used_mb": 12123.0
}

--- Concurrency = 5 ---
{
  "n": 20,
  "errors": 0,
  "error_rate_pct": 0.0,
  "avg_latency_s": 6.21,
  "p95_latency_s": 8.551,
  "avg_ttft_s": 0.23,
  "throughput_req_s": 0.772,
  "tokens_per_sec": 81.3,
  "wall_time_s": 25.91,
  "concurrency": 5,
  "gpu_util_pct": 100.0,
  "vra